# glass4: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('20_glass4_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,77.091,71.333,90.000,63.333,99.50,81.417,87.333,98.256,74.212
smote,89.148,75.952,76.167,83.333,97.25,90.292,87.778,98.839,82.550
bsmote,89.502,80.000,83.333,83.333,98.00,90.667,86.111,98.756,84.751
ros,89.376,78.571,80.833,83.333,97.75,90.542,82.611,97.703,83.974
adasyn,89.276,77.024,77.667,83.333,97.50,90.417,85.611,98.717,83.150
casb,89.376,78.571,80.833,83.333,97.75,90.542,81.706,97.453,83.974
casb_relaxed,89.479,79.905,84.167,83.333,98.00,90.667,85.873,98.673,84.692
casb_edited,90.958,78.095,77.500,86.667,97.25,91.958,85.040,98.596,84.527
casb_ros,89.250,77.143,78.333,83.333,97.50,90.417,82.040,97.537,83.197


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,86.597,82.667,93.333,76.667,99.500,88.083,83.322,93.224,84.632
smote,84.660,63.889,56.667,76.667,95.012,85.839,67.381,96.655,74.274
bsmote,89.913,67.167,56.667,86.667,95.012,90.839,69.278,97.474,78.540
ros,87.714,61.000,48.571,83.333,94.024,88.679,65.878,92.307,74.357
adasyn,91.542,69.500,59.571,90.000,94.762,92.381,65.883,94.502,80.521
casb,84.767,64.444,57.333,76.667,95.262,85.964,65.862,92.307,74.606
casb_relaxed,84.660,63.889,56.667,76.667,95.012,85.839,66.865,96.813,74.274
casb_edited,83.871,54.667,42.667,76.667,93.274,84.971,66.865,96.813,69.269
casb_ros,87.933,62.667,50.952,83.333,94.518,88.926,66.433,92.474,75.300


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,59.057,52.190,65.833,46.667,98.75,72.708,85.028,98.083,55.624
smote,91.487,83.000,85.667,86.667,98.25,92.458,87.278,98.884,87.243
bsmote,80.315,73.333,79.000,75.000,98.50,86.750,89.667,98.917,76.824
ros,85.534,75.548,84.333,76.667,98.50,87.583,90.333,99.083,80.541
adasyn,90.530,72.762,71.500,86.667,96.50,91.583,87.556,98.923,81.646
casb,76.054,67.190,74.167,68.333,98.25,83.292,82.556,98.140,71.622
casb_relaxed,86.184,76.833,74.333,85.000,98.25,91.625,85.111,98.596,81.509
casb_edited,90.804,75.714,75.667,86.667,97.00,91.833,84.222,98.352,83.259
casb_ros,74.364,66.000,74.000,65.000,98.50,81.750,86.333,98.589,70.182


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,23.094,20.000,40.000,13.333,98.000,55.667,56.526,89.112,21.547
smote,90.810,60.310,46.167,90.000,93.037,91.518,61.311,89.496,75.560
bsmote,74.106,54.429,47.000,73.333,94.768,84.051,59.803,89.175,64.267
ros,90.495,59.024,45.238,90.000,92.299,91.149,60.156,90.709,74.759
adasyn,85.483,55.238,43.833,80.000,93.043,86.521,58.251,89.936,70.361
casb,85.550,55.714,44.667,80.000,93.287,86.643,58.652,88.431,70.632
casb_relaxed,82.229,55.476,43.500,81.667,93.530,87.599,59.520,88.642,68.853
casb_edited,91.246,63.571,50.500,90.000,94.024,92.012,62.408,88.604,77.409
casb_ros,87.423,58.905,49.000,83.333,93.530,88.432,58.254,88.431,73.164


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,61.460,56.548,72.292,50.000,98.938,74.469,78.052,94.669,59.004
smote,89.026,70.788,66.167,84.167,95.887,90.027,75.937,95.969,79.907
bsmote,83.459,68.732,66.500,79.583,96.570,88.077,76.215,96.080,76.096
ros,88.280,68.536,64.744,83.333,95.643,89.488,74.745,94.951,78.408
adasyn,89.208,68.631,63.143,85.000,95.451,90.226,74.325,95.520,78.919
casb,83.937,66.480,64.250,77.083,96.137,86.610,72.194,94.083,75.208
casb_relaxed,85.638,69.026,64.667,81.667,96.198,88.932,74.342,95.681,77.332
casb_edited,89.220,68.012,61.583,85.000,95.387,90.194,74.634,95.591,78.616
casb_ros,84.742,66.179,63.071,78.750,96.012,87.381,73.265,94.258,75.461


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,90.958,bsmote,89.502,1.456,WIN
1,f1,casb_relaxed,79.905,bsmote,80.000,-0.095,LOSS
2,precision,casb_relaxed,84.167,bsmote,83.333,0.833,WIN
3,recall,casb_edited,86.667,smote,83.333,3.333,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,87.933,adasyn,91.542,-3.609,LOSS
1,f1,casb,64.444,adasyn,69.500,-5.056,LOSS
2,precision,casb,57.333,adasyn,59.571,-2.238,LOSS
3,recall,casb_ros,83.333,adasyn,90.000,-6.667,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,90.804,smote,91.487,-0.683,LOSS
1,f1,casb_relaxed,76.833,smote,83.000,-6.167,LOSS
2,precision,casb_edited,75.667,smote,85.667,-10.000,LOSS
3,recall,casb_edited,86.667,smote,86.667,0.000,TIE


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,89.220,adasyn,89.208,0.012,WIN
1,f1,casb_relaxed,69.026,smote,70.788,-1.762,LOSS
2,precision,casb_relaxed,64.667,bsmote,66.500,-1.833,LOSS
3,recall,casb_edited,85.000,adasyn,85.000,0.000,TIE


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.570088,0.400000,0.500000,0.333333,0.975000,0.654167,0.755556,0.975000,0.485044,None
1,42,1,svm,smote,0.50,0.961769,0.666667,0.500000,1.000000,0.925000,0.962500,0.805556,0.983333,0.814218,None
2,42,1,svm,bsmote,0.50,0.961769,0.666667,0.500000,1.000000,0.925000,0.962500,0.638889,0.975000,0.814218,None
3,42,1,svm,ros,0.50,0.961769,0.666667,0.500000,1.000000,0.925000,0.962500,0.638889,0.975000,0.814218,None
4,42,1,svm,adasyn,1.00,0.961769,0.666667,0.500000,1.000000,0.925000,0.962500,0.638889,0.975000,0.814218,None
5,42,1,svm,casb,0.50,0.961769,0.666667,0.500000,1.000000,0.925000,0.962500,0.638889,0.975000,0.814218,None
6,42,1,svm,casb_relaxed,0.50,0.961769,0.666667,0.500000,1.000000,0.925000,0.962500,0.638889,0.975000,0.814218,None
7,42,1,svm,casb_edited,0.50,0.961769,0.666667,0.500000,1.000000,0.925000,0.962500,0.638889,0.975000,0.814218,None
8,42,1,svm,casb_ros,0.50,0.961769,0.666667,0.500000,1.000000,0.925000,0.962500,0.638889,0.975000,0.814218,None
9,42,1,knn,original,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,0.809524,0.966667,0.808248,None
